# asbplayer Whisper GPU service

Run every cell in a Google Colab GPU runtime to create a temporary, authenticated Whisper subtitle service for asbplayer. Keep this notebook private: it downloads the video audio and transcribes it in this runtime. The service and its subtitle cache disappear when the Colab runtime stops.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then reconnect and rerun this notebook.'
print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
import os
from getpass import getpass

# Enter a new long random secret, then enter the same value in asbplayer Settings.
auth_token = getpass('Create a Whisper service token: ').strip()
assert len(auth_token) >= 24, 'Use a secret with at least 24 characters.'
os.environ['ASBPLAYER_WHISPER_AUTH_TOKEN'] = auth_token
os.environ['ASBPLAYER_WHISPER_DEVICE'] = 'cuda'

In [ ]:
repository = 'https://github.com/bensnhunt/asbplayer.git'
branch = 'feature/local-whisper-subtitle-generation'
!git clone --depth 1 --branch {branch} {repository} /content/asbplayer
!python -m pip install --upgrade /content/asbplayer/scripts/whisper-server

In [ ]:
import subprocess
import time
from pathlib import Path

# Do not leave Uvicorn's stdout in an unread pipe: its access logs can
# fill the pipe and block the service, which makes Cloudflare return 524.
server_log_path = Path('/content/asbplayer-whisper-server.log')
server_log = server_log_path.open('w', buffering=1)

server = subprocess.Popen(
    ['asbplayer-whisper-server', '--host', '127.0.0.1', '--port', '8767'],
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)
if server.poll() is not None:
    server_log.flush()
    raise RuntimeError(server_log_path.read_text())
print('Whisper service started with CUDA defaults.')
print(f'View service logs with: !tail -f {server_log_path}')

In [ ]:
import re
import urllib.request
from queue import Empty, Queue
from threading import Thread

# A Cloudflare quick tunnel provides the HTTPS address required by the extension.
cloudflared_path = '/content/cloudflared'
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    cloudflared_path,
)
os.chmod(cloudflared_path, 0o755)
tunnel = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:8767'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Keep draining cloudflared after its URL has been found for the same reason.
tunnel_log_path = Path('/content/asbplayer-whisper-tunnel.log')
tunnel_log = tunnel_log_path.open('w', buffering=1)
tunnel_lines = Queue()

def drain_tunnel_output():
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_log.write(line)
        tunnel_log.flush()
        tunnel_lines.put(line)

Thread(target=drain_tunnel_output, daemon=True).start()

tunnel_url = None
for _ in range(60):
    try:
        line = tunnel_lines.get(timeout=1)
    except Empty:
        continue
    print(line, end='')
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, 'Cloudflare Tunnel did not provide a URL. Rerun this cell.'
print('\nPaste this HTTPS URL into asbplayer Settings > Streaming Video > Whisper subtitle service:')
print(tunnel_url)
print('Paste the secret entered above as the Whisper Service Token, then choose Save Whisper service.')
print(f'View tunnel logs with: !tail -f {tunnel_log_path}')

## Keep it private

Do not share the tunnel URL or service token. Stop the runtime when finished; this ends the service and removes its temporary cache. If the Colab runtime disconnects, run the notebook again and update both the tunnel URL and token in asbplayer.